# Banc d'essai des descripteurs audio

Objectif : recommander par l'audio. Ce notebook compare quatre facons de
decrire l'audio d'un morceau, **en audio seul** (sans genre), sur tout le
catalogue. Le genre ne sert que d'etiquette pour evaluer.

| | Variante | Source |
|---|---|---|
| V0 | descripteurs de 2021, la reference | `data/df_recommendation.csv` |
| V1 | audio lu correctement (mono 22 050 Hz), memes 63 colonnes, un MFCC par coefficient | `python -m pipeline features-v2` |
| V2 | V1 + chroma, contraste, tonnetz et poly decrits bande par bande (109 colonnes) | idem |
| V3 | modele pre-entraine CLAP de LAION (`larger_clap_general` : musique, parole, sons ; Apache 2.0), 512 dimensions, reference | `python -m pipeline embeddings` |
| V4 | V1 + contraste et poly decrits bande par bande (81 colonnes) | `features-v2` |
| Vp | V2 avec des poids par famille, a modifier soi-meme (partie 3) | `features-v2` |

V4 vient d'un test des poids par famille (2026-10-09, `docs/recommandation.md`) :
dans V2, la chroma par bande fait perdre 5 points (elle decrit surtout la
tonalite) ; le contraste et le poly par bande en font gagner.

Les fichiers de V1, V2 et V3 sont dans `%LOCALAPPDATA%\museme\out\`. Contexte et
decisions : `docs/recommandation.md`, partie 7.

1. Chargement
2. Controles de vraisemblance
3. Espaces de similarite
4. Mesures
5. Detail par genre
6. Exemples
7. Courbe de duree
8. Ecoute a l'aveugle

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn import preprocessing
from sklearn.metrics.pairwise import cosine_similarity

sys.path.insert(0, "..")
from pipeline import ecoute
from pipeline.extract import preview_hash
from pipeline.features_v2 import V1_COLUMNS, V2_COLUMNS

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)

OUT = Path(os.environ["LOCALAPPDATA"]) / "museme" / "out"

## 1. Chargement

In [2]:
df = pd.read_csv("../data/df_recommendation.csv")
v2 = pd.read_csv(OUT / "features_v2.csv").set_index("row").sort_index()
clap = pd.read_csv(OUT / "embeddings_clap.csv").set_index("row").sort_index()
colonnes_clap = [c for c in clap.columns if c.startswith("clap")]

# Morceaux presents dans toutes les variantes (extrait telechargeable et calcule)
communs = df.index[df.index.isin(v2.index) & df.index.isin(clap.index)]

pd.DataFrame(
    {"morceaux": [len(df), len(v2), len(clap), len(communs)]},
    index=["catalogue", "V1 / V2 calcules", "V3 calcules", "communs (evalues)"],
)

,morceaux
catalogue,4724
V1 / V2 calcules,4724
V3 calcules,4724
communs (evalues),4724


In [3]:
# Duree des extraits analyses (secondes)
v2.loc[communs, "duree"].describe().round(1).to_frame("duree (s)").T

,count,mean,std,min,25%,50%,75%,max
duree (s),4724.0,29.1,3.0,14.0,30.0,30.0,30.0,37.2


## 2. Controles de vraisemblance

En 2021, l'extrait est lu 4 fois trop lentement : le tempo et le centroide
spectral sont fausses. V1 doit donner des valeurs plausibles.

In [4]:
pd.DataFrame({
    "tempo V0 (2021)": df.loc[communs, "tempo"].describe(),
    "tempo V1": v2.loc[communs, "tempo"].describe(),
    "centroide V0": df.loc[communs, "spectral_centroid_mean"].describe(),
    "centroide V1 (Hz)": v2.loc[communs, "spectral_centroid_mean"].describe(),
}).round(1)

,tempo V0 (2021),tempo V1,centroide V0,centroide V1 (Hz)
count,4724.0,4724.0,4724.0,4724.0
mean,117.5,123.4,3537.3,2434.5
std,14.7,20.0,846.3,581.0
min,43.8,55.0,0.0,0.0
25%,112.3,107.7,3093.0,2100.1
50%,123.0,123.0,3649.5,2495.4
75%,129.2,136.0,4121.2,2816.3
max,184.6,184.6,5821.3,4919.0


In [5]:
# Tempo : part des morceaux par tranche de 10 BPM
pd.DataFrame({
    "V0 (2021)": (df.loc[communs, "tempo"] // 10 * 10).value_counts(normalize=True).sort_index(),
    "V1": (v2.loc[communs, "tempo"] // 10 * 10).value_counts(normalize=True).sort_index(),
}).fillna(0).round(3)

,V0 (2021),V1
tempo,,
40.0,0.000,0.000
50.0,0.001,0.000
60.0,0.011,0.000
70.0,0.013,0.001
80.0,0.043,0.041
90.0,0.067,0.111
100.0,0.076,0.106
110.0,0.275,0.178
120.0,0.454,0.274


In [6]:
# MFCC : en 2021, mfcc{k} = k premiers coefficients ensemble, d'ou des colonnes
# correlees a 1,00. En V1, un coefficient par colonne.
pd.DataFrame({
    "V0 (2021)": [df.loc[communs, "mfcc10_var"].corr(df.loc[communs, "mfcc20_var"]),
                  df.loc[communs, "mfcc2_mean"].corr(df.loc[communs, "mfcc20_mean"])],
    "V1": [v2.loc[communs, "mfcc10_var"].corr(v2.loc[communs, "mfcc20_var"]),
           v2.loc[communs, "mfcc2_mean"].corr(v2.loc[communs, "mfcc20_mean"])],
}, index=["corr mfcc10_var / mfcc20_var", "corr mfcc2_mean / mfcc20_mean"]).round(2)

,V0 (2021),V1
corr mfcc10_var / mfcc20_var,1.00,0.59
corr mfcc2_mean / mfcc20_mean,0.88,-0.25


## 3. Espaces de similarite

Comme dans l'app : chaque colonne est centree-reduite, puis on compare par
cosinus. V3 aussi : bruts, les vecteurs CLAP occupent un cone etroit (cosinus
~0,98 entre deux morceaux quelconques) ; centres, ils se comparent comme les
autres. Pour chaque morceau, on garde ses 10 plus proches voisins, lui-meme
exclu.

In [7]:
X = {}
X["V0"] = preprocessing.scale(df.loc[communs, V1_COLUMNS])
X["V1"] = preprocessing.scale(v2.loc[communs, V1_COLUMNS])
X["V2"] = preprocessing.scale(v2.loc[communs, V2_COLUMNS])
X["V3"] = preprocessing.scale(clap.loc[communs, colonnes_clap])
V4_COLUMNS = V1_COLUMNS + [c for c in V2_COLUMNS if c.startswith(("spectral_contrast_b", "polyfeat_b"))]
X["V4"] = preprocessing.scale(v2.loc[communs, V4_COLUMNS])

# Vp : V2 avec un poids par famille de colonnes (prefixe du nom), a modifier.
# 1 = inchange, 0 = colonne retiree. Le premier prefixe qui correspond s'applique.
poids = {
    "chroma_stft_b": 0.3,        # chroma par bande (12 notes) : decrit surtout la tonalite
    "tonnetz_b": 1.0,
    "spectral_contrast_b": 1.0,
    "polyfeat_b": 1.0,
    "mfcc": 1.0,
    "tempo": 1.0,
}
w = np.array([next((p for prefixe, p in poids.items() if c.startswith(prefixe)), 1.0) for c in V2_COLUMNS])
X["Vp"] = preprocessing.scale(v2.loc[communs, V2_COLUMNS]) * w

voisins = {}
for nom in X:
    S = cosine_similarity(X[nom])
    np.fill_diagonal(S, -np.inf)
    voisins[nom] = np.argsort(-S, axis=1)[:, :10]

pd.DataFrame({nom: [X[nom].shape[1]] for nom in X}, index=["colonnes"])

C:\Users\Travail\OneDrive\Bureau\Dossiers\Projets\museme\.venv\Lib\site-packages\sklearn\preprocessing\_data.py:275: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(


,V0,V1,V2,V3,V4,Vp
colonnes,63,63,109,512,81,109


## 4. Mesures

Pour chaque morceau et ses 10 voisins :

- **meme artiste** : un autre titre du meme artiste est-il parmi les 10 ? (titre
  identique exclu ; seulement pour les morceaux dont l'artiste a d'autres titres) ;
- **meme genre** : part des 10 voisins du meme `Main_Genre` (au hasard : ~10 %) ;
- **eloigne** : part des voisins d'un genre non voisin sur le cercle (Pop et
  morceaux sans genre ne comptent jamais comme eloignes).

Intervalles de confiance a 95 % par bootstrap sur les morceaux. La colonne
« ecart avec V0 » compare les variantes morceau par morceau.

In [8]:
artiste = df.loc[communs, "Artist"].to_numpy()
titre = df.loc[communs, "Song"].str.lower().to_numpy()
genre = df.loc[communs, "Main_Genre"].to_numpy()
a_genre = pd.notna(genre)
autres_titres = pd.Series(titre).groupby(artiste).transform("nunique").to_numpy() >= 2

CERCLE = ["Asian Music", "R&B", "African Music", "Reggae", "Rap/Hip Hop", "Latin Music",
          "Brazilian Music", "Jazz", "Classical", "Folk", "Country", "Alternative", "Rock",
          "Metal", "Electro", "Dance"]
position = pd.Series(genre).map({g: k for k, g in enumerate(CERCLE)}).to_numpy()

scores = {}
for nom, V in voisins.items():
    ecart = np.abs(position[V] - position[:, None]) % 16
    ecart = np.minimum(ecart, 16 - ecart)
    scores[nom] = pd.DataFrame({
        "meme_artiste": ((artiste[V] == artiste[:, None]) & (titre[V] != titre[:, None])).any(axis=1),
        "meme_genre": (genre[V] == genre[:, None]).mean(axis=1),
        "eloigne": np.nan_to_num(ecart >= 2).mean(axis=1),
    })

print(f"{len(communs)} morceaux evalues ; {autres_titres.sum()} dont l'artiste a d'autres titres ; {a_genre.sum()} avec un genre")

4724 morceaux evalues ; 4066 dont l'artiste a d'autres titres ; 4697 avec un genre


In [9]:
rng = np.random.default_rng(0)
lignes = []
for mesure, masque in [("meme_artiste", autres_titres), ("meme_genre", a_genre), ("eloigne", a_genre)]:
    reference = scores["V0"].loc[masque, mesure].to_numpy(dtype=float)
    for nom in scores:
        v = scores[nom].loc[masque, mesure].to_numpy(dtype=float)
        tirages = rng.integers(0, len(v), size=(1000, len(v)))
        moyennes = v[tirages].mean(axis=1)
        ecarts = (v - reference)[tirages].mean(axis=1)
        lignes.append({
            "mesure": mesure,
            "variante": nom,
            "valeur": f"{v.mean():.1%} [{np.percentile(moyennes, 2.5):.1%} ; {np.percentile(moyennes, 97.5):.1%}]",
            "ecart avec V0 (points)": f"{100 * (v - reference).mean():+.1f} [{100 * np.percentile(ecarts, 2.5):+.1f} ; {100 * np.percentile(ecarts, 97.5):+.1f}]",
        })
mesures = pd.DataFrame(lignes)
mesures.pivot(index="variante", columns="mesure", values="valeur")

mesure,eloigne,meme_artiste,meme_genre
variante,,,
V0,34.2% [33.4% ; 34.9%],32.2% [30.9% ; 33.6%],25.2% [24.5% ; 25.8%]
V1,30.6% [29.7% ; 31.4%],39.8% [38.4% ; 41.3%],28.8% [28.2% ; 29.5%]
V2,30.7% [29.9% ; 31.5%],34.5% [33.3% ; 36.1%],27.5% [26.9% ; 28.2%]
V3,17.5% [16.8% ; 18.2%],61.7% [60.2% ; 63.1%],46.4% [45.5% ; 47.2%]
V4,30.2% [29.5% ; 31.1%],41.8% [40.3% ; 43.3%],29.5% [28.8% ; 30.2%]
Vp,30.6% [29.8% ; 31.4%],38.7% [37.3% ; 40.2%],28.2% [27.6% ; 28.9%]


In [10]:
mesures.pivot(index="variante", columns="mesure", values="ecart avec V0 (points)")

mesure,eloigne,meme_artiste,meme_genre
variante,,,
V0,+0.0 [+0.0 ; +0.0],+0.0 [+0.0 ; +0.0],+0.0 [+0.0 ; +0.0]
V1,-3.6 [-4.3 ; -2.9],+7.6 [+6.1 ; +9.1],+3.6 [+3.1 ; +4.2]
V2,-3.5 [-4.1 ; -2.8],+2.3 [+0.9 ; +3.9],+2.3 [+1.7 ; +3.0]
V3,-16.7 [-17.4 ; -16.0],+29.5 [+27.8 ; +31.2],+21.2 [+20.3 ; +22.0]
V4,-3.9 [-4.6 ; -3.3],+9.6 [+8.1 ; +11.1],+4.3 [+3.7 ; +4.9]
Vp,-3.6 [-4.2 ; -3.0],+6.5 [+5.0 ; +8.1],+3.0 [+2.5 ; +3.6]


## 5. Detail par genre

In [11]:
# Part des morceaux qui retrouvent un autre titre de leur artiste, selon leur genre
detail = pd.DataFrame({nom: scores[nom]["meme_artiste"][autres_titres].groupby(genre[autres_titres]).mean()
                       for nom in scores})
detail["morceaux"] = pd.Series(genre[autres_titres]).value_counts()
detail.sort_values("morceaux", ascending=False).style.format("{:.0%}", subset=list(scores))

,V0,V1,V2,V3,V4,Vp,morceaux
Pop,19%,31%,23%,53%,31%,25%,637
Alternative,27%,36%,27%,57%,37%,32%,565
Rock,37%,50%,44%,66%,52%,49%,553
Electro,22%,19%,19%,48%,21%,21%,462
Dance,25%,30%,28%,54%,30%,30%,322
Brazilian Music,50%,62%,57%,81%,64%,62%,255
R&B,21%,29%,21%,63%,33%,30%,219
Metal,52%,64%,62%,67%,64%,65%,192
Country,34%,42%,32%,78%,46%,40%,134
Latin Music,59%,69%,69%,78%,72%,69%,127


In [12]:
# Part des voisins du meme genre, selon le genre du morceau
detail = pd.DataFrame({nom: scores[nom]["meme_genre"][a_genre].groupby(genre[a_genre]).mean()
                       for nom in scores})
detail["morceaux"] = pd.Series(genre[a_genre]).value_counts()
detail.sort_values("morceaux", ascending=False).style.format("{:.0%}", subset=list(scores))

,V0,V1,V2,V3,V4,Vp,morceaux
Pop,24%,24%,23%,37%,25%,22%,746
Alternative,21%,24%,22%,36%,24%,22%,638
Rock,33%,36%,33%,46%,36%,35%,610
Electro,29%,33%,33%,50%,34%,33%,582
Dance,23%,32%,27%,43%,30%,30%,400
Brazilian Music,25%,32%,32%,61%,34%,32%,270
R&B,12%,13%,12%,35%,15%,14%,242
Metal,31%,38%,42%,48%,39%,39%,201
Rap/Hip Hop,22%,36%,30%,52%,35%,30%,195
Country,16%,16%,24%,67%,19%,19%,149


## 6. Exemples : les 5 premiers voisins

In [13]:
exemples = [
    ("4 Minutes (feat. Justin Timberlake & Timbaland)", "Madonna"),
    ("Boom Clap", "Charli XCX"),
    ("Dona Cila", "Maria Gadú"),
    ("Bad Moon Rising", "Creedence Clearwater Revival"),
    ("Next To You (feat. R. Kelly)", "Ciara"),
]
chanson = df.loc[communs, "Song"].to_numpy()
for s, a in exemples:
    trouve = np.flatnonzero((chanson == s) & (artiste == a))
    if len(trouve) == 0:
        print(f"{s} ({a}) : absent des morceaux evalues")
        continue
    i = trouve[0]
    print(f"\n{s} — {a} ({genre[i]})")
    display(pd.DataFrame({nom: [f"{artiste[j]} — {chanson[j][:28]} ({str(genre[j])[:5]})" for j in voisins[nom][i][:5]]
                          for nom in voisins}))


4 Minutes (feat. Justin Timberlake & Timbaland) — Madonna (Pop)


,V0,V1,V2,V3,V4,Vp
0,Kylie Minogue — Password (Pop),Lindstrøm — nummer fire en - 12” ver...,Miley Cyrus — Different Religion (fe...,RAYE — WHERE IS MY HUSBAND! (Pop),Lindstrøm — nummer fire en - 12” ver...,Miley Cyrus — Different Religion (fe...
1,Twisted Sister — Don't Let Me Down (...,Charli xcx — Camera (Pop),Covenant — Tour de force (Elect),Dua Lipa — Levitating (feat. DaBaby)...,Charli xcx — Camera (Pop),Covenant — Tour de force (Elect)
2,Romeo Santos — Dardos (Pop),Taylor Swift — Opalite (Pop),Radio Slave — Far Away / Augen der N...,Tate McRae — Sports car (Pop),Travis Scott — FE!N (feat. Playboi C...,Troye Sivan — Party (Pop)
3,Collective Soul — In A Moment (Rock),Kim Petras — Autobahn (Pop),Jorn Van Deynhoven — Punisher (Mixed...,Justin Bieber — Baby (Pop),Britney Spears — Womanizer (Pop),Lindstrøm — nummer fire en - 12” ver...
4,Ratt — Lay It Down (Rock),Britney Spears — Womanizer (Pop),VNV Nation — Aeroscope (Alter),Nelly Furtado — Maneater (Dance),Kim Petras — Autobahn (Pop),Charli xcx — Camera (Pop)



Boom Clap — Charli XCX (Pop)


,V0,V1,V2,V3,V4,Vp
0,MONGOL800 — 月灯りの下で (Rock),Solange — Where Do We Go (R&B),Jax Jones — All Day And Night (Dance),Bebe Rexha — Ferrari (Pop),Solange — Where Do We Go (R&B),Jax Jones — All Day And Night (Dance)
1,Christina Aguilera — Lady Marmalade ...,USHER — My Boo (Pop),Jung Kook — Seven (feat. Latto) (Exp...,Krewella — Pass the Love Around (Dance),Jax Jones — All Day And Night (Dance),PARTYNEXTDOOR — Nothing Easy to Plea...
2,JAŸ-Z — Empire State Of Mind (Rap/H),Jax Jones — All Day And Night (Dance),PARTYNEXTDOOR — Nothing Easy to Plea...,Namie Amuro — Break It (Dance),m-flo — Come Back To Me (Asian),Jung Kook — Seven (feat. Latto) (Exp...
3,Jason Derulo — Encore (Pop),Said the Sky — Just Us (Elect),Jah Sun — Brand New (Regga),JoJo — FAB. (feat. Remy Ma) (Pop),PARTYNEXTDOOR — Nothing Easy to Plea...,Solange — Where Do We Go (R&B)
4,OneRepublic — Counting Stars (Pop),Jung Kook — Seven (feat. Latto) (Exp...,Next — Butta Love (R&B),Selena Gomez — Same Old Love (Pop),Forró do Muido — Ela É Um Lixo / Cha...,K. Michelle — V.S.O.P. (R&B)



Dona Cila — Maria Gadú (Brazilian Music)


,V0,V1,V2,V3,V4,Vp
0,Nostalgia 77 — Beautiful Lie (Jazz),MYMP — Only Reminds Me of You (Pop),Eman Lam — 最幸運的人 (Asian),Eman Lam — 只怕不夠時間看你白頭 (Asian),MYMP — Only Reminds Me of You (Pop),Adriana Calcanhotto — A Fábrica Do P...
1,Billie Eilish — What Was I Made For?...,Róisín Murphy — Primitive (Elect),Adriana Calcanhotto — A Fábrica Do P...,Yeng Constantino — Di Na Ganun (Alter),Justin Lo — Loving You (Asian),Jimmy Bondoc — I Believe (Pop)
2,Anvil — What a Wonderful World (Metal),Toni Braxton — How Could an Angel Br...,MADKID — 出ていってよ (Asian),Brandy — Truthfully (Pop),Adriana Calcanhotto — A Fábrica Do P...,MYMP — Only Reminds Me of You (Pop)
3,Metronomy — Upset My Girlfriend (Alter),Justin Lo — Loving You (Asian),Jimmy Bondoc — I Believe (Pop),Maria Gadú — Taregué (Brazi),Toni Braxton — How Could an Angel Br...,Eman Lam — 最幸運的人 (Asian)
4,Air — All I Need (feat. Beth Hirsc (...,Adriana Calcanhotto — A Fábrica Do P...,Toni Braxton — How Could an Angel Br...,Kris Lawrence — If I Ever Fall in Lo...,Adele — Easy On Me (Pop),Toni Braxton — How Could an Angel Br...



Bad Moon Rising — Creedence Clearwater Revival (Rock)


,V0,V1,V2,V3,V4,Vp
0,Elmore James — Done Somebody Wrong (...,The Underground Youth — Revolution R...,Top Cats — Sometimes in the Rain (Pop),The Clean — Flowers (Folk),The Underground Youth — Revolution R...,Elmore James — Done Somebody Wrong (...
1,WITCH — Havoc (Afric),Billie Holiday — Riffin' the Scotch ...,Elmore James — Done Somebody Wrong (...,The Rolling Stones — (I Can't Get No...,Lord Huron — Ends of the Earth (Rock),Waylon Jennings — Are You Ready for ...
2,WITCH — Sweet Sixteen (Rock),Venom — In League with Satan (Metal),The Underground Youth — Revolution R...,The Byrds — She Don't Care About Tim...,Billie Holiday — Riffin' the Scotch ...,The Underground Youth — Revolution R...
3,Monophonics — In Your Brain (R&B),The Strokes — Barely Legal (Rock),The Stone Roses — Bye Bye Bad Man - ...,Allah-Las — Every Girl (Rock),Venom — In League with Satan (Metal),The Clean — Big Soft Punch (Alter)
4,Cake — Open Book (Pop),Ara Ketu — Coragem De Amor (Pop),The Clean — Big Soft Punch (Alter),The Kinks — Picture Book - 2018 Ster...,Damien Jurado — Nothing Is The News ...,The Black Angels — Empire (Alter)



Next To You (feat. R. Kelly) — Ciara (Pop)


,V0,V1,V2,V3,V4,Vp
0,Cody Simpson — Summertime of Our Liv...,Ciara — The Title (Pop),Yolanda Adams — Time After Time (R&B),Yolanda Adams — Time After Time (R&B),Ciara — The Title (Pop),Yolanda Adams — Time After Time (R&B)
1,Andrés Calamaro — Cuando te conocí (...,Yolanda Adams — Time After Time (R&B),Ciara — The Title (Pop),Ciara — The Title (Pop),Yolanda Adams — Time After Time (R&B),Ciara — The Title (Pop)
2,Osees — Canopnr '74 (Alter),Brandy — Truthfully (Pop),Ariana Grande — positions (Pop),K. Michelle — My Life (R&B),Pharrell Williams — Know Who You Are...,De La Rose — La Monda (Latin)
3,Rivermaya — Elesi (Pop),Sia — Cheap Thrills (feat. Sean Pa (...,Brandy — Love Is on My Side (Rock),Brandy — Truthfully (Pop),Brandy — Truthfully (Pop),Brandy — Love Is on My Side (Rock)
4,Tracy Chapman — Talkin' Bout a Revol...,De La Rose — La Monda (Latin),Ariana Grande — 7 rings (Pop),Sevyn Streeter — Been a Minute (feat...,De La Rose — La Monda (Latin),Ariana Grande — 7 rings (Pop)


## 7. Courbe de duree

Faut-il plus de 30 s d'audio ? Pour 1 000 morceaux, les vecteurs sont
recalcules sur une fenetre de 5 a 25 s placee au hasard dans l'extrait :

```powershell
foreach ($w in 5, 10, 15, 20, 25) {
  .\.venv\Scripts\python.exe -m pipeline features-v2 --window $w --rows 1000
  .\.venv\Scripts\python.exe -m pipeline embeddings --window $w --rows 1000
}
```

Pour chaque duree : la fenetre retrouve-t-elle son propre morceau (vecteur
calcule sur tout l'extrait) parmi tout le catalogue, et quelle part de ses 10
recommandations est identique a celles de l'extrait entier ? Si les valeurs
plafonnent avant 30 s, analyser plus longtemps n'apporterait presque rien.

In [14]:
lignes = []
for w in [5, 10, 15, 20, 25]:
    for nom, fichier, colonnes in [("V1", f"features_v2_w{w}.csv", V1_COLUMNS),
                                   ("V2", f"features_v2_w{w}.csv", V2_COLUMNS),
                                   ("V4", f"features_v2_w{w}.csv", V4_COLUMNS),
                                   ("V3", f"embeddings_clap_w{w}.csv", colonnes_clap)]:
        if not (OUT / fichier).exists():
            continue
        f = pd.read_csv(OUT / fichier).set_index("row")
        f = f[f.index.isin(communs)]
        complet = clap if nom == "V3" else v2
        echelle = preprocessing.StandardScaler().fit(complet.loc[communs, colonnes])
        Q = echelle.transform(f[colonnes])
        ici = communs.get_indexer(f.index)
        S = cosine_similarity(Q, X[nom])
        propre = S[np.arange(len(Q)), ici]
        rang = (S > propre[:, None]).sum(axis=1) + 1
        S[np.arange(len(Q)), ici] = -np.inf
        top = np.argsort(-S, axis=1)[:, :10]
        identiques = np.mean([len(set(top[k]) & set(voisins[nom][ici[k]])) / 10 for k in range(len(Q))])
        lignes.append({"variante": nom, "fenetre (s)": w, "morceaux": len(Q),
                       "retrouve en 1er": (rang == 1).mean(), "rang median": np.median(rang),
                       "recos identiques a 30 s": identiques})
courbe = pd.DataFrame(lignes)
courbe

,variante,fenetre (s),morceaux,retrouve en 1er,rang median,recos identiques a 30 s
0,V1,5,1000,0.876,1.0,0.4114
1,V2,5,1000,0.909,1.0,0.3991
2,V4,5,1000,0.878,1.0,0.3935
3,V3,5,1000,0.787,1.0,0.4033
4,V1,10,1000,0.945,1.0,0.5467
5,V2,10,1000,0.973,1.0,0.5506
6,V4,10,1000,0.958,1.0,0.5438
7,V1,15,1000,0.967,1.0,0.6396
8,V2,15,1000,0.983,1.0,0.6493
9,V4,15,1000,0.976,1.0,0.6452


## 8. Ecoute a l'aveugle

Une vingtaine de morceaux : ceux que tu connais bien (a completer ci-dessous),
plus un morceau tire au sort par genre. Pour chacun, les 5 premiers voisins
des variantes choisies sont melanges, doublons retires, sans dire qui les
propose. La page `ecoute.html` s'ouvre dans le navigateur ; ses reponses
s'exportent dans `ecoute_reponses.csv` (dossier Telechargements).

In [15]:
# Morceaux que tu connais bien : (titre, artiste), a completer
connus = list(exemples)

# Variantes a departager a l'oreille : V0 et les deux meilleures de la partie 4
variantes_ecoute = ["V0", "V4", "V3"]

In [16]:
rng = np.random.default_rng(1)
requetes = []
for s, a in connus:
    trouve = np.flatnonzero((chanson == s) & (artiste == a))
    if len(trouve):
        requetes.append(trouve[0])
for g in pd.Series(genre[a_genre]).value_counts().index:
    if len(requetes) >= 20:
        break
    candidats_genre = [i for i in np.flatnonzero(genre == g) if i not in requetes]
    requetes.append(rng.choice(candidats_genre))

cle = []
for q, i in enumerate(requetes, 1):
    proposes = {}
    for nom in variantes_ecoute:
        for rang, j in enumerate(voisins[nom][i][:5], 1):
            proposes.setdefault(j, []).append(f"{nom}:{rang}")
    ordre = rng.permutation(list(proposes))
    for c, j in enumerate(ordre, 1):
        cle.append({"q": q, "c": c, "position": j, "requete": i, "variantes": " ".join(proposes[j])})
cle = pd.DataFrame(cle)
cle.to_csv(OUT / "ecoute_cle.csv", index=False)

audio = df.loc[communs, "Preview"].map(lambda u: f"../previews/{preview_hash(u)}.mp3").to_numpy()
page_requetes = pd.DataFrame({"q": range(1, len(requetes) + 1), "Song": chanson[requetes],
                              "Artist": artiste[requetes], "audio": audio[requetes]})
page_candidats = pd.DataFrame({"q": cle["q"], "c": cle["c"], "Song": chanson[cle["position"]],
                               "Artist": artiste[cle["position"]], "audio": audio[cle["position"]]})
ecoute.write_page(page_requetes, page_candidats, OUT / "ecoute.html")
print(f"{len(requetes)} morceaux, {len(cle)} propositions a noter -> {OUT / 'ecoute.html'}")
page_requetes[["q", "Song", "Artist"]]

20 morceaux, 277 propositions a noter -> C:\Users\Travail\AppData\Local\museme\out\ecoute.html


,q,Song,Artist
0,1,4 Minutes (feat. Justin Timberlake &...,Madonna
1,2,Boom Clap,Charli XCX
2,3,Dona Cila,Maria Gadú
3,4,Bad Moon Rising,Creedence Clearwater Revival
4,5,Next To You (feat. R. Kelly),Ciara
5,6,Buwan - Acoustic Uso Version,The Itchyworms
6,7,Using an Online Psychic to Know Abou...,The Slow Death
7,8,No Lives Matter,Body Count
8,9,Interesting,Sara Landry
9,10,Crank It Up (feat. Akon),David Guetta


In [17]:
# Apres l'ecoute : lecture des reponses exportees
reponses_csv = Path.home() / "Downloads" / "ecoute_reponses.csv"
if reponses_csv.exists():
    notes = pd.read_csv(reponses_csv).merge(cle, on=["q", "c"])
    lignes = []
    for nom in variantes_ecoute:
        n = notes[notes["variantes"].str.contains(nom + ":")]
        par_requete = n.groupby("q")["note"].mean().to_numpy()
        tirages = rng.integers(0, len(par_requete), size=(1000, len(par_requete)))
        moyennes = par_requete[tirages].mean(axis=1)
        lignes.append({"variante": nom, "notes": len(n),
                       "proche": (n["note"] == 2).mean(), "sans rapport": (n["note"] == 0).mean(),
                       "note moyenne (0-2)": f"{par_requete.mean():.2f} [{np.percentile(moyennes, 2.5):.2f} ; {np.percentile(moyennes, 97.5):.2f}]"})
    display(pd.DataFrame(lignes))
else:
    print(f"Pas encore de reponses : {reponses_csv}")

Pas encore de reponses : C:\Users\Travail\Downloads\ecoute_reponses.csv
